## **Technical Analysis of Shared Dataset**

#### Importing required libraries and tools

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

#### Importing the csv file from the local source (desktop)

Initial look of the data set - the data types and the structure of the set through the use of .head and .info function

In [ ]:
data=pd.read_csv(r'C:\Users\Surya\Documents\Free2Move.csv')
data.head()
data.info()

#### Creating additional colums for analysis

Converted the format of the STARTED_TIME AND FINISHED_TIME from object to date time to hlep in further analysis. Added collums such as used_time and charge_used derived from the difference in start and end time of rental, the difference in start and end charge of the rental respectively.

In [ ]:
data['STARTED_TIME'] = pd.to_datetime(data['STARTED_TIME'])
data['FINISHED_TIME'] = pd.to_datetime(data['FINISHED_TIME'])
data['used_time'] =data['FINISHED_TIME'] - data['STARTED_TIME']  
data["charge_used"] = data["CHARGELEVELSTART"] - data["CHARGELEVELEND"]

##### Created a seperate dataframe for outliers 
For ease of analysis and to also use it later for deeper analysis if need be, found **133 outliers** in the data with usage time of more than a day

In [ ]:
outliers=data[data['used_time'].dt.days > 0]

#### Check for null data and their relationship

Found the presence of null data through initial analysis and now wanted to deep dive how does the presence of null data in few of these collums (especially CHARGELEVELSTART and CHARGELEVELEND) affect data in other collums.

**1.CHARGELEVELSTART**  
For the CHARGELEVELSTART variable we assign a missing df and compare the mean of all the boolean values generated by the function .isna - so we get an idea about how many of the other variables are being affected by NA value of CHARGELEVELSTART. 


In [ ]:
data['missing'] = data['CHARGELEVELSTART'].isna()
data.groupby('missing').mean(numeric_only=True)

From the table above it canbe seen that the null value in CHARGELEVELSTART has a drastic impact on the SERVICERENTAL parameter whereas with others not that evident.

**Removing the "missing" table**

In [ ]:
data = data.drop(columns=['missing'])

**2.CHARGELEVELEND**  
For the CHARGELEVELSTART variable we assign a missing df and compare the mean of all the boolean values generated by the function .isna - so we get an idea about how many of the other variables are being affected by NA value of CHARGELEVELSTART. 

In [ ]:
data['missing'] = data['CHARGELEVELEND'].isna()
data.groupby('missing').mean(numeric_only=True)

From the table above it can be seen that the null value in CHARGELEVELEND has a minor impact on the other parameter hence for this analysis I am not considering this to be vital.

##### Create seperate dataframs
Created seperate df for customer data and service data by the collum SERVICERENTAL. As understood from the above analysis that null values are mostly caused by service agents utilising the vehicle. 

In [ ]:
customer_data = data[data['SERVICERENTAL'] == False]
service_data  = data[data['SERVICERENTAL'] == True]

Created another df from customer data where the customer is not charging the vehicle on their own (assuming - used fro shorter duarations)

In [ ]:
customer_data_no_charge = customer_data[customer_data["CHARGED"] == False].copy()
customer_data_no_charge

###### Mini Test 1
WIth this data set (customer_data_no_charge) I am trying to see if there is any relation between the use time of the rental and the battery consumed (charge_used) assuming that more the vehicle is used greater the charge depleted. I am converting the used time to hours and then using the .corr() function to get the correlation coeff. beween the variables. Also I am droping the null elemnets and negative numbers.

In [ ]:
customer_data_no_charge["used_time_h"] = (customer_data_no_charge["used_time"].dt.total_seconds() / 3600)
corr = customer_data_no_charge[["used_time_h", "charge_used"]].corr()
customer_data_no_charge = customer_data_no_charge.dropna()
customer_data_no_charge = customer_data_no_charge[customer_data_no_charge["charge_used"] >= 0]
corr                                           

From the table above we can infer that the charge used has very little corelation with used time. Proving my hypothesis to be wrong.

In [ ]:
plt.figure()
plt.scatter(
    customer_data_no_charge["used_time_h"],
    customer_data_no_charge["charge_used"],
    alpha=0.1
)
plt.xlabel("Used Time-hours")
plt.ylabel("Charge Used %")
plt.xlim(0, 10)
plt.title("Battery Used vs Rental Duration")
plt.show()

The plot of the relationship between Battery used and Rental duration (limit to less than 10h) - we can see that the scatter plot is scattered and doesnt form any paticular pattern, just that people tend to use vehicles for less than 2 hours and expand less than 20% of the battery. 

##### Mini Test 2
I wanted to check if the Charge start level had any relationship with the battery used. Assumption being that battery consumption(percentage wise) will be lesser for vehicles having higher battery start level (ex. battery% will go down slower from 90-80 than from 40-30)

In [ ]:
plt.figure()
plt.scatter(
    customer_data_no_charge["CHARGELEVELSTART"],
    customer_data_no_charge["charge_used"],
    alpha=0.1
)
plt.xlabel("Starting_Charge")
plt.ylabel("Charge Used (%)")
plt.title("Battery Used vs Charge Start Level")
plt.show()

Through the plot I was able to only infer that there was less usage of vehicles below the 20% startcharge mark and people mostly used it for less than 20%

**Removing any null values and droped the missing values collum.**

In [ ]:
customer_data = customer_data.drop(columns=['missing'])
customer_data = customer_data.dropna()
customer_data.info()
customer_data.head()

###### Mini Test 3 

Created a seperate df cut from the customer data in splits of 10s - chargelevel at the start of the ride and determined their frequency. This shows the relationship between the charge level start brackets and the frequency of ride rentals. Also shown is the probability distribution of the frequency.  

In [ ]:
bins = range(0, 110, 10)
bucket = pd.cut(customer_data["CHARGELEVELSTART"], bins=bins)
cus_charge = bucket.value_counts().sort_index().reset_index()
cus_charge.columns = ["split", "count"]

cus_charge["probability %"] = cus_charge["count"] / cus_charge["count"].sum() * 100
cus_charge

From the Table it is visible that **below 20% of charge the frequency and the probability of a rental being triggered is low** but for higher charge splits such as 80-90 and 90-100 the probability is a lot higher.

Below is a histogram showing **the frequency of use vs the charge start level**, the above statement can be seen reflected on the figure.

In [ ]:
plt.hist(customer_data["CHARGELEVELSTART"], bins)
plt.xlabel("Charge Level (%)")
plt.ylabel("Count")
plt.title("Distribution of Charge Start Level")
plt.show()

Below is a Bar chart showing **The Probability of Customer Renting vs The Charge Start Level**.


In [ ]:
plt.bar(cus_charge["split"].astype(str), cus_charge["probability %"])
plt.xlabel("Charge Level Range (%)")
plt.ylabel("Probability")
plt.title("Probability of Customer Renting at Various Charge Levels")
plt.xticks(rotation=45)
plt.show()


##### Mini Test 4
Did a quick check on the number of unique vehicles rented - **51**  
Created a df 'Model' withthe various vehicle ID and the frequency of their usage.  Also shown is the probability distribution of the frequency.

In [ ]:
unique_models = data['VEHICLE_ID'].nunique()
unique_models
Model = data["VEHICLE_ID"].value_counts().reset_index().sort_index()
Model.columns = ["VEHICLE_ID", "count"]
Model["probability %"] = Model["count"] / Model["count"].sum() * 100
Model

**Inference** - Can be see that paticular vehicle "dc753ee97d00780d5b36379e37ee78bf" has been used thae least and has abnormaly low chance of being used.

##### Mini Test 5

Broke down the end charge level into splits of 10% and compared with the usage of both customer and service datasets

In [ ]:
bucket = pd.cut(customer_data["CHARGELEVELEND"], bins=bins)
bucket2 = pd.cut(service_data["CHARGELEVELEND"], bins=bins)

CHARGELEVELEND_service = bucket2.value_counts().reset_index().sort_index()
CHARGELEVELEND_customer = bucket.value_counts().reset_index().sort_index()

CHARGELEVELEND_service.columns = ["CHARGELEVELEND", "count"]
CHARGELEVELEND_customer.columns = ["CHARGELEVELEND", "count"]
CHARGELEVELEND_customer = CHARGELEVELEND_customer.sort_values("CHARGELEVELEND").reset_index(drop=True)
CHARGELEVELEND_service = CHARGELEVELEND_service.sort_values("CHARGELEVELEND").reset_index(drop=True)

CHARGELEVELEND_service

The Inference from the sevice data set is that there is a lot of movement by agents when the charge is lowe than 20% and higher than 90%  
   
And for the data set below customers arent leaving the vehicle at very less charge after they use it, reducing the stress on agents

In [ ]:
CHARGELEVELEND_customer

##### Mini Test 6

Checking howmany customers charge their vehicles when they rent it.


In [ ]:
customer_charging = customer_data["CHARGED"].value_counts().reset_index()
customer_charging.columns = ["CHARGED", "count"]

customer_charging

##### Mini Test 7

Created bins of time duration to check if people charge the vehicles and at what ranges they do 

In [ ]:
bins_d = pd.to_timedelta([0, 1, 2, 4, 8, 24], unit="h")
labels = ["0–1", "1–2", "2–4", "4–8", "8+"]

customer_data["duration_bin"] = pd.cut(customer_data["used_time"],bins=bins_d,labels=labels,right=False)
charging_vs_duration = (customer_data.groupby(["duration_bin", "CHARGED"]).size().unstack(fill_value=0))
charging_vs_duration["percent_charged"] = (charging_vs_duration[True] /charging_vs_duration.sum(axis=1)) * 100
charging_vs_duration

Inferred that higher the duration of rental higher the percentage of people charging the vehicle

##### Mini Test 8

Similar test as above but to check if ctarting charge of the vehicle has any impact on people charging


In [ ]:
customer_data["start_charge_bin"] = pd.cut(customer_data["CHARGELEVELSTART"],bins=bins)
chargelevel_vs_charged = (customer_data.groupby(["start_charge_bin", "CHARGED"]).size().unstack(fill_value=0))
chargelevel_vs_charged["percent_charged"] = (chargelevel_vs_charged[True] /chargelevel_vs_charged.sum(axis=1)) * 100

chargelevel_vs_charged

Inffered- Probability of People charging is more when the battery percent is below 30%

##### Mini Test 9

This Test conducts a matrix evaluation on duration and start charge and how these two are correlated when it comes to people using the vehicle

In [ ]:
chargelevel_vs_duration = (customer_data.groupby(["start_charge_bin", "duration_bin"]).size().unstack(fill_value=0))
chargelevel_vs_duration

Inference- People preffer renting vehicles with higher charge even if they use it for less than an hour there seem to be a clear demarkation that can be drawn diagonaly from 30%-40% to 8+ hours at 31 users.The line above which people dont preffer and below which people tend to use the vehicle.

###### Mini Test 10
This next experiment explores the relationship between vehicleIDs and various other factors.  Also set a low charge metrics to see vehicles that are constantly undercharged<20% this is converted to a low cost probability. The combined table is shown below

In [ ]:
avg_charge_per_vehicle = (customer_data.groupby("VEHICLE_ID")["CHARGELEVELSTART"].mean().reset_index())
low_charge_vehicle = (customer_data[customer_data["CHARGELEVELSTART"] <20].groupby("VEHICLE_ID").size()
                      .reset_index(name="low_charge_rentals"))
low_charge_vehicle = low_charge_vehicle.sort_values("low_charge_rentals").reset_index(drop=True)
final_df = (low_charge_vehicle.merge(avg_charge_per_vehicle, on="VEHICLE_ID", how="left").merge(Model, on="VEHICLE_ID", how="left"))
final_df = final_df.sort_values("low_charge_rentals").reset_index(drop=True)
final_df["low_charge_probability"] = (final_df["low_charge_rentals"] / final_df["count"]) * 100
final_df


Inference- One paticualar vehicle is constantly under performing 'dc753ee97d00780d5b36379e37ee78bf' other vehicles have a comparable performance. For low charge probability the vehicle '1c347a2ffbbdfb70197160e8e77c9a49' is being constantly undercharged, althogh this percent isnt considerable, this may affect the battery of the vehicle in the longer run. 

These we all the test that were conducted and the inference from these test have be transfered to the Presentation that was submitted along with this document.